# Preet Sports — Free Colab AI Cricket Image Worker
Run this notebook manually when you want to produce a video. Free Colab sessions are temporary and may disconnect. Do not use this as an always-on service.

**Setup:** create a strong shared token, set it as a Colab Secret named `PREET_COLAB_TOKEN`, and set the same value as Render `COLAB_WORKER_TOKEN`. Copy the public Cloudflare URL printed in the last cell into Render `COLAB_IMAGE_WORKER_URL` (without a trailing slash). Never commit the token.

In [ ]:
!pip -q install --upgrade diffusers transformers accelerate safetensors fastapi uvicorn pillow

In [ ]:
import os, io, base64, secrets, subprocess, threading, time, requests
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field
from diffusers import AutoPipelineForText2Image
import torch
from PIL import Image
from google.colab import userdata

TOKEN = userdata.get('PREET_COLAB_TOKEN')
if not TOKEN or len(TOKEN) < 32:
    raise RuntimeError('Create Colab Secret PREET_COLAB_TOKEN with at least 32 random characters.')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU is available in this Colab session. Select a GPU runtime and reconnect.')

MODEL_ID = 'stabilityai/sdxl-turbo'
pipe = AutoPipelineForText2Image.from_pretrained(MODEL_ID, torch_dtype=torch.float16, variant='fp16')
pipe.to('cuda')
pipe.enable_attention_slicing()
print('Loaded', MODEL_ID, 'on', torch.cuda.get_device_name(0))

In [ ]:
app = FastAPI(title='Preet Sports Colab Image Worker')

class GenerateRequest(BaseModel):
    prompts: list[str] = Field(min_length=8, max_length=12)
    width: int = Field(default=1024, ge=512, le=1024)
    height: int = Field(default=576, ge=512, le=1024)
    steps: int = Field(default=4, ge=1, le=4)

@app.get('/health')
def health():
    return {'ok': True, 'model': MODEL_ID, 'gpu': torch.cuda.get_device_name(0)}

@app.post('/generate')
def generate(req: GenerateRequest, authorization: str = Header(default='')):
    if not secrets.compare_digest(authorization, 'Bearer ' + TOKEN):
        raise HTTPException(status_code=401, detail='Unauthorized')
    images=[]
    for prompt in req.prompts:
        result = pipe(prompt=prompt, num_inference_steps=req.steps, guidance_scale=0.0,
                      width=req.width, height=req.height)
        image = result.images[0].convert('RGB')
        buf=io.BytesIO(); image.save(buf, format='JPEG', quality=90)
        images.append({'data': base64.b64encode(buf.getvalue()).decode('ascii'), 'mime_type':'image/jpeg'})
        del result
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return {'images': images, 'count': len(images), 'model': MODEL_ID}

def run_api():
    import uvicorn
    uvicorn.run(app, host='0.0.0.0', port=7860, log_level='warning')

threading.Thread(target=run_api, daemon=True).start()
time.sleep(3)
print(requests.get('http://127.0.0.1:7860/health').json())

In [ ]:
# Create a temporary public HTTPS tunnel. The API still requires your secret bearer token.
import platform
if platform.system() != 'Linux': raise RuntimeError('Expected a Colab Linux runtime.')
subprocess.run(['bash','-lc', 'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared'], check=True)
proc = subprocess.Popen(['/usr/local/bin/cloudflared','tunnel','--url','http://localhost:7860','--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
public_url=None
deadline=time.time()+90
while time.time()<deadline:
    line=proc.stdout.readline()
    if 'trycloudflare.com' in line:
        import re
        match=re.search(r'https://[a-zA-Z0-9.-]+\\.trycloudflare\\.com', line)
        if match:
            public_url=match.group(0); break
    if proc.poll() is not None: break
if not public_url: raise RuntimeError('Could not start Cloudflare tunnel. Inspect the cell output and rerun.')
print('COLAB_IMAGE_WORKER_URL =', public_url)
print('Set this URL in Render. Keep this notebook running during video generation.')
print('Health check:', requests.get(public_url+'/health', timeout=30).status_code)